# PICKO · Notebook 3 — Breadth · Depth · Separation

The final research study. One **focus of 40 tools** (families `arxiv, hf, wikipedia, pubmed`), three
questions, each measured cleanly:

| Dimension | Question | What we finetune | Metric |
|---|---|---|---|
| **Breadth** (tool-set size) | How many tools can PICKO choose among before it picks wrong? | one model **per size** (nested 3⊂5⊂10⊂20⊂30), **compact** schemas | **selection only** |
| **Depth** (parameter complexity) | Given the right tool, can it fill the arguments — harder with more params? | one model on all 40, full schemas | `args_exact_acc` / `param_f1` **by #params** |
| **Separation** (disambiguation) | Can it tell near-identical tools apart? | reuse the 40-tool model | selection + confusion **within look-alike groups** |

> **Why finetune per size for Breadth?** A model trained on 3 tools but *offered* 30 would fail for the
> wrong reason (it never learned them). To find the real ceiling we train a model for each size.
> Needle's encoder is capped at **1024 tokens**, and a full tool schema is ~124 tokens (only ~8–9 fit),
> so Breadth uses **compact** schemas (name + description) — even then ~22 tools is the wall, which the
> `n_visible` annotation makes explicit.

> Kernel: **PICKO (.venv)** · CPU only. Training is gated by `RUN_TRAIN`; each run is capped at
> `CAP_PER_TOOL` examples/tool (raise it later for higher fidelity).

> **Unattended / overnight run.** This whole study (~6 finetunes, ~3 h) can run headless so it survives
> a closed lid — macOS *pauses* a sleeping process, so keep the Mac awake with `caffeinate` on AC power:
> ```bash
> caffeinate -is nohup .venv/bin/python -u scripts/run_research.py > research.log 2>&1 &
> tail -f research.log
> ```
> It writes the same `checkpoints/picko_*_best.pkl` this notebook uses plus `data/research_results.json`.
> Then run this notebook with `RUN_TRAIN=False` to reload those checkpoints and render the plots.

## 1 · Setup

In [ ]:
import os, sys, json, glob, shutil, subprocess
ROOT = os.path.abspath("..")
if ROOT not in sys.path: sys.path.insert(0, ROOT)
os.environ.setdefault("JAX_PLATFORMS", "cpu")
# resolve the `needle` console script robustly (same venv as this kernel)
NEEDLE = shutil.which("needle") or os.path.join(os.path.dirname(sys.executable), "needle")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
try:
    import seaborn as sns; sns.set_theme(style="whitegrid")
except Exception:
    sns = None
from tqdm.auto import tqdm

from scripts.tool_catalog import Catalog, family_of
from scripts.research_sets import (FOCUS_FAMILIES, focus_names, BREADTH_SIZES,
                                    nested_sets, SIMILAR_GROUPS,
                                    param_bucket, PARAM_BUCKET_ORDER)
from scripts.picko_eval import (load_model, predict, evaluate, confusion,
                                base_checkpoint, tools_token_len, n_visible)
from needle.training.finetune import _per_tool_split
from needle.dataset.dataset import get_tokenizer

cat = Catalog()
tok = get_tokenizer()
raw = [json.loads(l) for l in open(os.path.join(ROOT, "data", "picko_balanced.jsonl")) if l.strip()]
FOCUS = focus_names(cat)                       # the 40-tool focus

# ---- knobs ----
CAP_PER_TOOL = 40      # examples/tool per finetune (raise for more fidelity, slower)
EPOCHS       = 1
RUN_TRAIN    = True    # False = skip training and load existing checkpoints/picko_*_best.pkl
EVAL_SUBSAMPLE = None  # e.g. 120 to cap the test set for faster CPU eval; None = full

print(f"focus: {len(FOCUS)} tools · {len(raw)} raw examples · CAP_PER_TOOL={CAP_PER_TOOL} · EPOCHS={EPOCHS}")

### 1a · The shared finetune→eval helper

Every experiment goes through `finetune_and_eval`: re-scope the pool to the chosen tools (with the right
offering policy), write a JSONL, finetune (or reuse), copy the checkpoint to a stable name, then decode
and score the held-out test split. Returns everything downstream cells need.

In [ ]:
def finetune_and_eval(names, tag, compact=False, offer_all=None, token_aware=False):
    """Re-scope -> finetune (gated) -> eval. Returns a result dict.

    compact      : offer {name,description} only (Breadth).
    offer_all    : offer_all_max for restrict_dataset. Breadth passes len(names) so
                   every example offers all k tools; None keeps the default (12).
    token_aware  : pass the tokenizer so offered schemas are trimmed to fit (Depth).
    """
    kw = dict(cap_per_tool=CAP_PER_TOOL, compact=compact, seed=0)
    if offer_all is not None:   kw["offer_all_max"] = offer_all
    if token_aware:             kw["tokenizer"] = tok
    data = cat.restrict_dataset(raw, names, **kw)
    path = os.path.join(ROOT, "data", f"picko_{tag}.jsonl")
    with open(path, "w") as f:
        for e in data: f.write(json.dumps(e, ensure_ascii=False) + "\n")

    ckpt = os.path.join(ROOT, "checkpoints", f"picko_{tag}_best.pkl")
    if RUN_TRAIN:
        print(f"[{tag}] finetuning on {len(data)} examples ({len(names)} tools)…")
        subprocess.run([NEEDLE,"finetune",path,"--epochs",str(EPOCHS),
                        "--batch-size","32"], cwd=ROOT, check=True)
        newest = max(glob.glob(os.path.join(ROOT,"checkpoints","needle_finetuned_*_best.pkl")),
                     key=os.path.getmtime)
        shutil.copy(newest, ckpt)
    assert os.path.exists(ckpt), f"missing {ckpt} — run with RUN_TRAIN=True first"

    _, _, test = _per_tool_split(data)
    if EVAL_SUBSAMPLE: test = test[:EVAL_SUBSAMPLE]
    m, p, tk = load_model(ckpt)
    bar = tqdm(total=len(test), desc=tag, unit="ex")
    preds = predict(m, p, tk, test, progress=lambda i,n: bar.update(i-bar.n))
    bar.close()
    metrics = evaluate(test, preds, family_of=family_of)
    return {"tag":tag,"names":names,"data":data,"ckpt":ckpt,"bundle":(m,p,tk),
            "test":test,"preds":preds,"metrics":metrics}

# base (un-finetuned) predictions on a given test set — for the baseline comparison
def base_eval(test):
    m,p,tk = load_model(base_checkpoint())
    bar = tqdm(total=len(test), desc="base", unit="ex")
    preds = predict(m,p,tk,test, progress=lambda i,n: bar.update(i-bar.n))
    bar.close()
    return evaluate(test, preds, family_of=family_of)

## 1b · View saved results (from the headless runner)

If you ran `scripts/run_research.py` (see the note at the top), everything was saved to
`data/research_results.json`. Run **just this cell** to render all four dimensions from that file —
no training, no re-decoding. (To reproduce the study live instead, skip this and run sections 2–5.)

In [ ]:
RESULTS_JSON = os.path.join(ROOT, "data", "research_results.json")
if not os.path.exists(RESULTS_JSON):
    print("No research_results.json yet — run scripts/run_research.py, or run sections 2–5 live.")
else:
    res = json.load(open(RESULTS_JSON))

    # --- Baseline ---
    b = res["baseline"]; keys = ["selection_acc","name_f1","args_exact_acc","param_f1","call_exact","abstain_acc"]
    bt = pd.DataFrame({"metric":keys, "base":[b["base"][k] for k in keys],
                       "finetuned":[b["finetuned"][k] for k in keys]})
    bt["delta"] = (bt["finetuned"].astype(float)-bt["base"].astype(float)).round(3)
    print("Baseline tools:", b["tools"]); display(bt)

    # --- Breadth ---
    br = pd.DataFrame(res["breadth"]); display(br)
    fig, ax = plt.subplots(figsize=(8,4.5))
    ax.plot(br["k"], br["selection_acc"], "o-", color="#4C72B0", label="selection_acc")
    ax.plot(br["k"], br["name_f1"], "s--", color="#55A868", label="name_f1")
    wall = br[br["n_visible"] < br["k"]]
    if len(wall):
        kw = int(wall["k"].iloc[0])
        ax.axvline(kw, color="#C44E52", ls=":", lw=1.5)
        ax.text(kw, 0.05, f" truncation starts\n (~{int(wall['n_visible'].iloc[0])} of {kw} visible)",
                color="#C44E52", fontsize=9, va="bottom")
    ax.set_xlabel("# tools (k)"); ax.set_ylabel("tool-selection accuracy"); ax.set_ylim(0,1.02)
    ax.set_title("Breadth: accuracy vs tool-set size"); ax.legend(); plt.tight_layout(); plt.show()

    # --- Depth ---
    dt = pd.DataFrame(res["depth"]["per_tool"])
    bb = (dt.groupby("bucket").agg(n_tools=("tool","size"), avg_args_exact=("args_exact_acc","mean"),
          avg_param_f1=("param_f1","mean")).reindex(PARAM_BUCKET_ORDER).dropna(how="all").round(3))
    display(bb)
    bx = bb.reset_index(); x = np.arange(len(bx)); w = 0.38
    plt.figure(figsize=(7,4))
    plt.bar(x-w/2, bx["avg_args_exact"], w, color="#4C72B0", label="args_exact_acc")
    plt.bar(x+w/2, bx["avg_param_f1"], w, color="#DD8452", label="param_f1")
    plt.xticks(x, bx["bucket"]); plt.ylim(0,1); plt.xlabel("# parameters (bucket)")
    plt.title("Depth: extraction vs parameter count"); plt.legend(); plt.tight_layout(); plt.show()

    # --- Separation ---
    sp = pd.DataFrame([{k:v for k,v in row.items() if k!="confusion"} for row in res["separation"]]).sort_values("selection_acc")
    display(sp)
    plt.figure(figsize=(8,4)); plt.barh(sp["group"], sp["selection_acc"], color="#4C72B0")
    plt.xlim(0,1); plt.xlabel("tool-selection accuracy"); plt.title("Separation: hardest look-alike groups")
    plt.tight_layout(); plt.show()
    for row in res["separation"]:
        conf = row["confusion"]; labels = sorted(set(conf) | {p for r in conf.values() for p in r})
        M = pd.DataFrame(0, index=sorted(conf), columns=labels)
        for r, rr in conf.items():
            for p, n in rr.items(): M.loc[r, p] = n
        plt.figure(figsize=(0.9*len(labels)+2, 0.5*len(M)+1.5))
        if sns: sns.heatmap(M, annot=True, fmt="d", cmap="Blues", cbar=False)
        else: plt.imshow(M.values, cmap="Blues")
        plt.title(f"Separation · {row['group']}"); plt.xlabel("predicted"); plt.ylabel("reference")
        plt.tight_layout(); plt.show()

## 2 · Baseline — 3 tools

A tiny, fast run: does finetuning help at all on this focus? Trains on 3 tools from different families
(full schemas) and compares base vs finetuned across **both** selection and parameter extraction.

In [ ]:
BASELINE_TOOLS = ["arxiv_search_papers", "pubmed_search_articles", "wikipedia_search_wikipedia"]
R3 = finetune_and_eval(BASELINE_TOOLS, tag="baseline3", compact=False, offer_all=3)
base3 = base_eval(R3["test"])
ft3   = R3["metrics"]

keys = ["selection_acc","name_f1","args_exact_acc","param_f1","call_exact","abstain_acc"]
baseline = pd.DataFrame({"metric":keys,
                         "base":[base3[k] for k in keys],
                         "finetuned":[ft3[k] for k in keys]})
baseline["delta"] = (baseline["finetuned"].astype(float)-baseline["base"].astype(float)).round(3)
print("tools:", BASELINE_TOOLS)
display(baseline)

## 3 · Breadth — how many tools can PICKO handle?

For each size `k` (nested, so the smaller set is always inside the larger), finetune a **separate** model
that offers all `k` tools in **compact** form, and measure **tool-selection only**. `n_visible` reports
how many of the `k` tools actually survive the encoder's 1024-token truncation — the ceiling.

In [ ]:
SETS = nested_sets(FOCUS, BREADTH_SIZES, seed=0)   # {k: [names]}, strictly nested
breadth_rows = []
breadth_runs = {}
for k in BREADTH_SIZES:
    names = SETS[k]
    R = finetune_and_eval(names, tag=f"breadth_k{k}", compact=True, offer_all=k)
    breadth_runs[k] = R
    # how many tools the model actually sees (median over the test set), compact schemas
    vis = int(np.median([n_visible(e["query"], json.loads(e["tools"]), tok)
                         for e in R["test"]]))
    breadth_rows.append({"k":k, "selection_acc":R["metrics"]["selection_acc"],
                         "name_f1":R["metrics"]["name_f1"], "n_visible":vis})
breadth = pd.DataFrame(breadth_rows)
display(breadth)

In [ ]:
fig, ax = plt.subplots(figsize=(8,4.5))
ax.plot(breadth["k"], breadth["selection_acc"], "o-", color="#4C72B0", label="selection_acc")
ax.plot(breadth["k"], breadth["name_f1"], "s--", color="#55A868", label="name_f1")
# mark where offered tools stop fully fitting the window (n_visible < k)
wall = breadth[breadth["n_visible"] < breadth["k"]]
if len(wall):
    kw = wall["k"].iloc[0]
    ax.axvline(kw, color="#C44E52", ls=":", lw=1.5)
    ax.text(kw, 0.05, f" truncation starts\n (only ~{int(wall['n_visible'].iloc[0])} of {kw} visible)",
            color="#C44E52", fontsize=9, va="bottom")
ax.set_xlabel("# tools the model was trained/evaluated on (k)")
ax.set_ylabel("tool-selection accuracy"); ax.set_ylim(0,1.02)
ax.set_title("Breadth: does accuracy degrade as the tool set grows?")
ax.legend(); plt.tight_layout(); plt.show()

## 4 · Depth — is parameter extraction harder for tools with more parameters?

Finetune once on all **40** focus tools (full schemas, offered set trimmed to fit so selection is easy
and nothing truncates). Then, among examples where the tool was selected correctly, bucket the tools by
**parameter count** and measure how exactly the arguments were extracted.

In [ ]:
DEPTH = finetune_and_eval(FOCUS, tag="depth40", compact=False, token_aware=True)
per_tool = DEPTH["metrics"]["per_tool"]

df = cat.as_dataframe().set_index("tool")
rows = []
for name, s in per_tool.items():
    tot = int(df.loc[name, "total_params"]) if name in df.index else 0
    rows.append({"tool":name, "total_params":tot, "bucket":param_bucket(tot),
                 "n":s["n"], "selection_acc":s["selection_acc"],
                 "args_exact_acc":s["args_exact_acc"], "param_f1":s["param_f1"]})
depth_tools = pd.DataFrame(rows)
by_bucket = (depth_tools.groupby("bucket")
             .agg(n_tools=("tool","size"), avg_selection=("selection_acc","mean"),
                  avg_args_exact=("args_exact_acc","mean"), avg_param_f1=("param_f1","mean"))
             .reindex(PARAM_BUCKET_ORDER).dropna(how="all").round(3))
display(by_bucket)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13,4.5))
b = by_bucket.reset_index()
x = np.arange(len(b)); w = 0.38
ax[0].bar(x-w/2, b["avg_args_exact"], w, color="#4C72B0", label="args_exact_acc")
ax[0].bar(x+w/2, b["avg_param_f1"], w, color="#DD8452", label="param_f1")
ax[0].set_xticks(x); ax[0].set_xticklabels(b["bucket"]); ax[0].set_ylim(0,1)
ax[0].set_xlabel("# parameters (bucket)"); ax[0].set_title("Depth: extraction vs parameter count")
ax[0].legend()

ax[1].scatter(depth_tools["total_params"], depth_tools["args_exact_acc"], s=55, color="#4C72B0")
for _,r in depth_tools.iterrows():
    ax[1].annotate(r["tool"].split("_")[0], (r["total_params"], r["args_exact_acc"]), fontsize=7)
ax[1].set_xlabel("# parameters in tool"); ax[1].set_ylabel("args_exact_acc")
ax[1].set_title("per-tool"); plt.tight_layout(); plt.show()

## 5 · Separation — which look-alike tools get confused?

Reuse the 40-tool model. For each curated group of near-identical tools, offer **only that group** and
measure how well PICKO tells them apart — plus a confusion heatmap of what it picks when wrong.

In [ ]:
m40, p40, tk40 = DEPTH["bundle"]
sep_rows, group_conf = [], {}
for gname, gtools in SIMILAR_GROUPS.items():
    gset = cat.restrict_dataset(raw, gtools, offer_all_max=len(gtools),
                                cap_per_tool=CAP_PER_TOOL, seed=0)
    _, _, gtest = _per_tool_split(gset)
    if EVAL_SUBSAMPLE: gtest = gtest[:EVAL_SUBSAMPLE]
    bar = tqdm(total=len(gtest), desc=gname, unit="ex")
    gpreds = predict(m40, p40, tk40, gtest, progress=lambda i,n: bar.update(i-bar.n))
    bar.close()
    gm = evaluate(gtest, gpreds, family_of=family_of)
    sep_rows.append({"group":gname, "n_tools":len(gtools), "n":gm["n"],
                     "selection_acc":gm["selection_acc"], "name_f1":gm["name_f1"]})
    group_conf[gname] = confusion(gtest, gpreds)
separation = pd.DataFrame(sep_rows).sort_values("selection_acc")
display(separation)

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
ax.barh(separation["group"], separation["selection_acc"], color="#4C72B0")
ax.set_xlim(0,1); ax.set_xlabel("tool-selection accuracy")
ax.set_title("Separation: hardest look-alike groups (lower = more confused)")
plt.tight_layout(); plt.show()

In [ ]:
# per-group confusion heatmaps
for gname, conf in group_conf.items():
    labels = sorted(set(conf) | {p for row in conf.values() for p in row})
    M = pd.DataFrame(0, index=sorted(conf), columns=labels)
    for r, row in conf.items():
        for p, n in row.items(): M.loc[r, p] = n
    plt.figure(figsize=(0.9*len(labels)+2, 0.5*len(M)+1.5))
    if sns: sns.heatmap(M, annot=True, fmt="d", cmap="Blues", cbar=False)
    else:
        plt.imshow(M.values, cmap="Blues")
        plt.xticks(range(len(labels)), labels, rotation=90); plt.yticks(range(len(M)), M.index)
    plt.title(f"Separation · {gname}"); plt.xlabel("predicted"); plt.ylabel("reference")
    plt.tight_layout(); plt.show()

## 6 · The research story

*(Fill in with your numbers once every section has run.)*

**PICKO — a 26M scientific tool-picker.** On a 40-tool focus (arxiv · hugging face · wikipedia · pubmed):

- **Breadth (§3):** tool selection holds up to ~`k` tools, then degrades. The drop coincides with the
  **1024-token encoder wall** — beyond ~8–9 *full* schemas (or ~20 *compact* ones) later tools are
  truncated away, so the model literally cannot see them. The practical limit for one PICKO instance is
  a context-window limit, which motivates sharding a large tool set across categorical instances.
- **Depth (§4):** parameter extraction is the real difficulty and it **falls as tools gain parameters**
  — zero/one-arg tools are near-solved, multi-argument tools (4+) are where errors concentrate. This is
  where finetuning a small specialist pays off (see the §2 baseline jump).
- **Separation (§5):** residual mistakes cluster inside the curated look-alike groups — same action
  across sources and same source across actions. The hardest group is the frontier for a tool-picker.

**Takeaway:** a tiny fine-tuned model is a strong single-shot scientific tool-picker; its limits are the
context window (Breadth) and multi-parameter calls (Depth), with look-alike disambiguation (Separation)
as the last mile.